# LeetCode 874 — Walking Robot Simulation bằng Python

Link đề: https://leetcode.com/problems/walking-robot-simulation/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Hash Table (bảng băm), Simulation (mô phỏng)

Skeleton LeetCode:

```python
class Solution:
    def robotSim(self, commands: List[int], obstacles: List[List[int]]) -> int:
        ...
```

Ghi chú: đây là bài **mô phỏng** thuần túy. Hai việc quyết định tốc độ và độ đúng: (1) kiểm tra "ô tiếp theo có chướng ngại không" trong `O(1)` (bảng băm thay vì quét danh sách) và (2) cập nhật khoảng cách lớn nhất **ở mỗi bước** (không chỉ ở cuối), vì robot có thể đi ra xa rồi quay về.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: robot bắt đầu ở `(0, 0)` hướng **bắc**; lệnh `-2` quay trái 90 độ, `-1` quay phải 90 độ, `1..9` đi thẳng từng ô một; gặp chướng ngại thì **đứng yên** và sang lệnh kế; trả về **bình phương khoảng cách Euclid lớn nhất** từ gốc tới mọi vị trí đã đi qua.
2. Nắm các công cụ Python cần thiết: `set` các tuple tọa độ, mảng hướng `dirs = (0, 1, 0, -1, 0)` (cặp `dirs[k], dirs[k + 1]`), phép chia lấy dư `% 4` để quay, số phức làm vectơ hướng, và `bisect` để nhảy tới chướng ngại gần nhất.
3. Hiểu nguyên lý mô phỏng: **trạng thái** = (vị trí, hướng); mỗi lệnh quay chỉ đổi hướng, mỗi lệnh đi tiến từng ô và dừng khi ô kế tiếp là chướng ngại.
4. Đi qua 4 cách: quét danh sách chướng ngại (brute), `set` các tuple + mảng hướng (cách nộp chính), số phức, và nhảy theo hàng/cột bằng `bisect`; đối chiếu bằng oracle mô phỏng độc lập (hướng bằng chữ `N/E/S/W`, khóa ô dạng chuỗi).
5. Tránh các lỗi kinh điển: đi liền `k` ô mà không kiểm tra chướng ngại giữa đường, chỉ lấy khoảng cách ở cuối, đảo hướng quay trái/phải, di chuyển rồi mới kiểm tra, và coi chướng ngại ở `(0, 0)` chặn ngay từ đầu.

## 2. Đề bài & ví dụ

Một robot trên mặt phẳng `XY` vô hạn bắt đầu ở `(0, 0)` và hướng về phía **bắc**. Robot nhận mảng số nguyên `commands`, mỗi phần tử là một lệnh:

- `-2`: quay trái `90` độ;
- `-1`: quay phải `90` độ;
- `1 <= k <= 9`: đi thẳng `k` đơn vị, **từng đơn vị một**.

Một số ô là **chướng ngại** `obstacles[i] = (xi, yi)`. Nếu robot đụng chướng ngại, nó **đứng yên ở ô liền kề chướng ngại** và chuyển sang lệnh kế tiếp.

Trả về **bình phương khoảng cách Euclid lớn nhất** từ gốc tới robot tại mọi điểm trên đường đi (khoảng cách `5` thì trả `25`).

Lưu ý:

- Có thể có chướng ngại ở `(0, 0)`: robot **bỏ qua** nó cho tới khi rời gốc, nhưng sau đó không thể quay lại `(0, 0)`.
- Bắc là `+Y`, đông là `+X`, nam là `-Y`, tây là `-X`.

**Ví dụ 1:** `commands = [4,-1,3]`, `obstacles = []` -> `25` (`(0,4)` rồi `(3,4)`: `9 + 16 = 25`).

**Ví dụ 2:** `commands = [4,-1,4,-2,4]`, `obstacles = [[2,4]]` -> `65` (đi bắc 4 tới `(0,4)`, quay phải, đi đông bị chặn bởi `(2,4)` nên dừng ở `(1,4)`, quay trái, đi bắc 4 tới `(1,8)`: `1 + 64 = 65`).

**Ví dụ 3:** `commands = [6,-1,-1,6]`, `obstacles = [[0,0]]` -> `36` (đi bắc 6 tới `(0,6)`, quay phải hai lần (hướng nam), đi nam bị chặn bởi `(0,0)` nên dừng ở `(0,1)`).

**Ràng buộc:**

- `1 <= commands.length <= 10^4`
- `commands[i]` là `-2`, `-1` hoặc số nguyên trong `[1, 9]`
- `0 <= obstacles.length <= 10^4`
- `-3 * 10^4 <= xi, yi <= 3 * 10^4`
- Đáp án đảm bảo nhỏ hơn `2^31`.

In [1]:
from typing import List
from bisect import bisect_left, bisect_right
import random, time

examples = [
    ([4, -1, 3], [], 25),
    ([4, -1, 4, -2, 4], [[2, 4]], 65),
    ([6, -1, -1, 6], [[0, 0]], 36),
]
for commands, obstacles, expected in examples:
    print(f"commands={commands} obstacles={obstacles} kỳ vọng={expected}")

commands=[4, -1, 3] obstacles=[] kỳ vọng=25
commands=[4, -1, 4, -2, 4] obstacles=[[2, 4]] kỳ vọng=65
commands=[6, -1, -1, 6] obstacles=[[0, 0]] kỳ vọng=36


## 3. Kiến thức nền

### 3.1. Mảng hướng và phép quay bằng `% 4`

Bốn hướng theo thứ tự **bắc, đông, nam, tây** (quay phải đi theo chiều này). Mảng `dirs = (0, 1, 0, -1, 0)` có độ dài 5: hướng `k` có vectơ `(dirs[k], dirs[k + 1])`:

| `k` | hướng | `(dx, dy)` |
|---|---|---|
| 0 | bắc | `(0, 1)` |
| 1 | đông | `(1, 0)` |
| 2 | nam | `(0, -1)` |
| 3 | tây | `(-1, 0)` |

- Quay phải: `k = (k + 1) % 4`.
- Quay trái: `k = (k + 3) % 4` (hay `(k - 1) % 4`; Python cho kết quả không âm).

In [2]:
dirs = (0, 1, 0, -1, 0)
names = ["bắc", "đông", "nam", "tây"]
k = 0
for cmd in (-1, -1, -2, -2, -2):
    k = (k + 1) % 4 if cmd == -1 else (k + 3) % 4
    print(f"lệnh {cmd:>2}: hướng {names[k]:>4} -> (dx, dy) = ({dirs[k]}, {dirs[k + 1]})")
print("(-1) % 4 =", (-1) % 4, "(Python luôn cho số không âm khi chia dư cho số dương)")

lệnh -1: hướng đông -> (dx, dy) = (1, 0)
lệnh -1: hướng  nam -> (dx, dy) = (0, -1)
lệnh -2: hướng đông -> (dx, dy) = (1, 0)
lệnh -2: hướng  bắc -> (dx, dy) = (0, 1)
lệnh -2: hướng  tây -> (dx, dy) = (-1, 0)
(-1) % 4 = 3 (Python luôn cho số không âm khi chia dư cho số dương)


### 3.2. `set` các tuple để kiểm tra chướng ngại `O(1)`

Tuple `(x, y)` là hashable. `s = {(x, y) for x, y in obstacles}` cho phép `(nx, ny) in s` trong `O(1)` trung bình, thay vì quét danh sách `O(m)`.

In [3]:
obstacles = [[2, 4], [-3, 5], [0, 0]]
s = {(x, y) for x, y in obstacles}
print(s, (2, 4) in s, (2, 5) in s)

{(2, 4), (-3, 5), (0, 0)} True False


### 3.3. Số phức làm vectơ hướng (tùy chọn)

Số phức `x + yj` biểu diễn điểm `(x, y)`. Hướng bắc là `1j`; **quay trái 90 độ** là nhân với `1j` (`1j -> -1 -> -1j -> 1`), quay phải là nhân với `-1j`. Cộng số phức là cộng vectơ.

In [4]:
d = 1j                                     # bắc
for step in range(4):
    print("hướng", d, "-> quay trái ->", d * 1j, "| quay phải ->", d * -1j)
    d *= 1j
pos = 0j
pos += 1j * 4          # đi bắc 4
pos += 1 * 3           # đi đông 3
print("vị trí:", pos, "| bình phương khoảng cách:", int(pos.real) ** 2 + int(pos.imag) ** 2)

hướng 1j -> quay trái -> (-1+0j) | quay phải -> (1-0j)
hướng (-1+0j) -> quay trái -> (-0-1j) | quay phải -> 1j
hướng (-0-1j) -> quay trái -> (1-0j) | quay phải -> (-1+0j)
hướng (1-0j) -> quay trái -> 1j | quay phải -> (-0-1j)
vị trí: (3+4j) | bình phương khoảng cách: 25


### 3.4. Bình phương khoảng cách trên đường thẳng đạt cực đại ở đầu mút

Hàm `x² + y²` dọc một đoạn thẳng là hàm **lồi** theo tham số, nên giá trị lớn nhất trên đoạn đạt ở **một trong hai đầu mút**. Do đó mỗi lệnh đi chỉ cần xét vị trí **sau khi đi** (đầu mút đầu đã xét từ lệnh trước); điều này được dùng trong Cách 4 (nhảy một lần). Các cách còn lại cập nhật sau từng bước nên cũng đúng.

## 4. Ý tưởng nền tảng / trực giác (tính tay)

Với ví dụ 2: `commands = [4,-1,4,-2,4]`, `obstacles = [[2,4]]`:

| Lệnh | Trạng thái sau lệnh | Ghi chú | `ans` |
|---|---|---|---|
| (bắt đầu) | `(0,0)` hướng bắc | | 0 |
| `4` | `(0,4)` | 4 bước bắc | `16` |
| `-1` | `(0,4)` hướng đông | quay phải | 16 |
| `4` | `(1,4)` | bước 1 tới `(1,4)` ổn; bước 2 sẽ tới `(2,4)` là chướng ngại nên dừng | `17` |
| `-2` | `(1,4)` hướng bắc | quay trái | 17 |
| `4` | `(1,8)` | 4 bước bắc | `65` |

Mô phỏng chỉ có `O(số lệnh * 9)` bước, rất nhỏ; vấn đề duy nhất là kiểm tra chướng ngại nhanh.

## 5. Cách 1 — Brute Force: quét danh sách chướng ngại ở mỗi bước

**Ý tưởng.** Mô phỏng đúng đề: ở mỗi bước nhìn ô kế tiếp `(nx, ny)`, nếu nó nằm trong **danh sách** `obstacles` (quét tuyến tính bằng `in`) thì dừng, ngược lại đi tới và cập nhật `ans`. Đúng nhưng chậm khi `obstacles` lớn.

In [5]:
def robot_brute(commands: List[int], obstacles: List[List[int]]) -> int:
    obs = [(x, y) for x, y in obstacles]            # danh sách, tra cứu O(m)
    dirs = (0, 1, 0, -1, 0)
    x = y = k = ans = 0
    for c in commands:
        if c == -2:
            k = (k + 3) % 4
        elif c == -1:
            k = (k + 1) % 4
        else:
            for _ in range(c):
                nx, ny = x + dirs[k], y + dirs[k + 1]
                if (nx, ny) in obs:                 # quét tuyến tính
                    break
                x, y = nx, ny
                ans = max(ans, x * x + y * y)
    return ans

for commands, obstacles, expected in examples:
    got = robot_brute(commands, obstacles)
    print(commands, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[4, -1, 3] -> 25 OK
[4, -1, 4, -2, 4] -> 65 OK
[6, -1, -1, 6] -> 36 OK


**Độ phức tạp và nút thắt.** `O(9 * n * m)`: với `n = 10^4` lệnh (mỗi lệnh đi 9 bước) và `m = 10^4` chướng ngại là `9 * 10^8` phép so sánh, quá chậm. Hãy xem thời gian khi `m` tăng:

In [6]:
rng = random.Random(874)
n_cmds = 2000
commands = [rng.choice([-2, -1] + list(range(1, 10))) for _ in range(n_cmds)]
for m in (10, 100, 1000, 3000):
    obstacles = [[rng.randint(-300, 300), rng.randint(-300, 300)] for _ in range(m)]
    t0 = time.perf_counter()
    robot_brute(commands, obstacles)
    print(f"m={m:>5}: {1000 * (time.perf_counter() - t0):9.1f} ms")

m=   10:       4.6 ms
m=  100:      20.0 ms


m= 1000:     160.9 ms


m= 3000:     303.5 ms


Thời gian tăng gần **tuyến tính theo `m`** (mỗi bước quét cả danh sách). Nút thắt: phép "ô này có chướng ngại không?" nên là `O(1)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Đưa chướng ngại vào bảng băm.** Với `set` các tuple `(x, y)`, mỗi phép kiểm tra là `O(1)`, tổng thời gian `O(C * n + m)` với `C = 9` (số bước tối đa của một lệnh) và `n` lệnh. Phần còn lại là mô phỏng đúng đề:

- giữ **vị trí** `(x, y)` và **chỉ số hướng** `k` (0 bắc, 1 đông, 2 nam, 3 tây);
- lệnh quay đổi `k` bằng `% 4`;
- lệnh đi: lặp tối đa `c` lần: tính ô kế tiếp, **nếu là chướng ngại thì `break`** (kết thúc lệnh), ngược lại di chuyển và cập nhật đáp án.

| Cách | Ý tưởng |
|---|---|
| 2 | `set` tuple + mảng `dirs` (nộp) |
| 3 | `set` các số phức; quay bằng nhân `1j` |
| 4 | Gộp chướng ngại theo hàng và cột, **nhảy** một lần tới ô sát chướng ngại bằng `bisect` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — `set` các tuple + mảng hướng (cách nộp chính)

**Giải thích từng đoạn.**

- `s = {(x, y) for x, y in obstacles}`: chướng ngại, tra `O(1)`.
- `dirs = (0, 1, 0, -1, 0)`: `(dirs[k], dirs[k + 1])` là hướng `k`.
- Lệnh quay: cập nhật `k`.
- Lệnh đi `c`: `for _ in range(c)`: ô kế `(nx, ny)`; nếu `(nx, ny) in s` thì `break`; ngược lại gán `x, y = nx, ny` và `ans = max(ans, x * x + y * y)`.
- Chướng ngại tại `(0, 0)`: robot xuất phát ở `(0, 0)` mà không bao giờ "đi vào" ô đó, nên nó bị bỏ qua lúc đầu; còn khi quay lại sát gốc thì ô kế tiếp `(0, 0)` nằm trong `s` và robot bị chặn, đúng theo đề.

In [7]:
def robot_set(commands: List[int], obstacles: List[List[int]]) -> int:
    dirs = (0, 1, 0, -1, 0)
    s = {(x, y) for x, y in obstacles}
    x = y = k = ans = 0
    for c in commands:
        if c == -2:
            k = (k + 3) % 4
        elif c == -1:
            k = (k + 1) % 4
        else:
            for _ in range(c):
                nx, ny = x + dirs[k], y + dirs[k + 1]
                if (nx, ny) in s:
                    break
                x, y = nx, ny
                ans = max(ans, x * x + y * y)
    return ans

for commands, obstacles, expected in examples:
    got = robot_set(commands, obstacles)
    print(commands, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[4, -1, 3] -> 25 OK
[4, -1, 4, -2, 4] -> 65 OK
[6, -1, -1, 6] -> 36 OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def robotSim(self, commands: List[int], obstacles: List[List[int]]) -> int:
        dirs = (0, 1, 0, -1, 0)
        s = {(x, y) for x, y in obstacles}
        ans = k = 0
        x = y = 0
        for c in commands:
            if c == -2:
                k = (k + 3) % 4
            elif c == -1:
                k = (k + 1) % 4
            else:
                for _ in range(c):
                    nx, ny = x + dirs[k], y + dirs[k + 1]
                    if (nx, ny) in s:
                        break
                    x, y = nx, ny
                    ans = max(ans, x * x + y * y)
        return ans

sol_set = Solution().robotSim
assert [sol_set(c, o) for c, o, _ in examples] == [e for _, _, e in examples]
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** thời gian `O(C * n + m)` với `C = 9` (số bước tối đa mỗi lệnh), `n = len(commands)`, `m = len(obstacles)`; bộ nhớ `O(m)`.

---

### 7.2. Cách 3 — Số phức làm vị trí và hướng

**Ý tưởng khác.** Vị trí `pos` và hướng `d` đều là số phức. Hướng ban đầu `d = 1j` (bắc). Quay trái: `d *= 1j`; quay phải: `d *= -1j`. Đi một bước: `nxt = pos + d`. Chướng ngại lưu dạng `complex(x, y)` trong `set`. Khoảng cách bình phương: `pos.real ** 2 + pos.imag ** 2` (số thực, nhưng giá trị nguyên và nhỏ nên chính xác).

**Giải thích từng đoạn.** Phép nhân `d * 1j` xoay vectơ 90 độ ngược chiều kim đồng hồ, chính là quay trái khi bắc là `+Y`.

In [9]:
def robot_complex(commands: List[int], obstacles: List[List[int]]) -> int:
    s = {complex(x, y) for x, y in obstacles}
    pos, d = 0j, 1j
    ans = 0
    for c in commands:
        if c == -2:
            d *= 1j
        elif c == -1:
            d *= -1j
        else:
            for _ in range(c):
                nxt = pos + d
                if nxt in s:
                    break
                pos = nxt
                ans = max(ans, int(pos.real) ** 2 + int(pos.imag) ** 2)
    return ans

for commands, obstacles, expected in examples:
    got = robot_complex(commands, obstacles)
    print(commands, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[4, -1, 3] -> 25 OK
[4, -1, 4, -2, 4] -> 65 OK
[6, -1, -1, 6] -> 36 OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def robotSim(self, commands: List[int], obstacles: List[List[int]]) -> int:
        s = {complex(x, y) for x, y in obstacles}
        pos, d = 0j, 1j
        ans = 0
        for c in commands:
            if c == -2:
                d *= 1j
            elif c == -1:
                d *= -1j
            else:
                for _ in range(c):
                    nxt = pos + d
                    if nxt in s:
                        break
                    pos = nxt
                    ans = max(ans, int(pos.real) ** 2 + int(pos.imag) ** 2)
        return ans

sol_complex = Solution().robotSim
assert [sol_complex(c, o) for c, o, _ in examples] == [e for _, _, e in examples]
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(C * n + m)` thời gian, `O(m)` bộ nhớ (phép nhân số phức có hằng số lớn hơn tuple một chút).

---

### 7.3. Cách 4 — Gộp chướng ngại theo hàng/cột và nhảy bằng `bisect`

**Ý tưởng khác.** Thay vì đi từng ô, với mỗi lệnh đi `c` ta tìm **chướng ngại gần nhất trên đường đi** rồi nhảy luôn tới vị trí cuối: `min(c, khoảng cách tới ô sát chướng ngại)`.

- `rows[y]`: danh sách **đã sắp xếp** các `x` của chướng ngại nằm trên hàng `y` (dùng khi đi đông/tây).
- `cols[x]`: danh sách đã sắp xếp các `y` của chướng ngại trên cột `x` (đi bắc/nam).
- Đi đông: `idx = bisect_right(rows[y], x)` (chướng ngại đầu tiên có `x_o > x`); nếu tồn tại thì giới hạn `x_o - 1`: `x = min(x + c, x_o - 1)`.
- Đi tây: `idx = bisect_left(rows[y], x) - 1` (chướng ngại cuối cùng có `x_o < x`); giới hạn `x_o + 1`: `x = max(x - c, x_o + 1)`.
- Bắc/nam tương tự với `cols[x]` và `y`.
- Chỉ cần cập nhật `ans` ở vị trí **cuối** lệnh (mục 3.4: cực đại ở đầu mút).

Ưu điểm: không phụ thuộc số bước `C` của mỗi lệnh (hữu ích nếu `C` lớn); nhược: nhiều trường hợp hơn, dễ sai.

**Giải thích từng đoạn.** Dùng `bisect_right` cho hướng dương (tìm phần tử `> x`) và `bisect_left` cho hướng âm (tìm phần tử `< x`) nên chướng ngại ngay tại vị trí hiện tại (chỉ có thể là `(0, 0)` lúc xuất phát) bị bỏ qua đúng như đề.

In [11]:
from collections import defaultdict

def robot_bisect(commands: List[int], obstacles: List[List[int]]) -> int:
    rows, cols = defaultdict(list), defaultdict(list)
    for ox, oy in obstacles:
        rows[oy].append(ox)
        cols[ox].append(oy)
    for lst in rows.values():
        lst.sort()
    for lst in cols.values():
        lst.sort()
    x = y = k = ans = 0
    for c in commands:
        if c == -2:
            k = (k + 3) % 4
        elif c == -1:
            k = (k + 1) % 4
        else:
            if k == 0:                                   # bắc: tăng y trên cột x
                lst = cols.get(x, [])
                idx = bisect_right(lst, y)
                y = min(y + c, lst[idx] - 1) if idx < len(lst) else y + c
            elif k == 2:                                 # nam: giảm y trên cột x
                lst = cols.get(x, [])
                idx = bisect_left(lst, y) - 1
                y = max(y - c, lst[idx] + 1) if idx >= 0 else y - c
            elif k == 1:                                 # đông: tăng x trên hàng y
                lst = rows.get(y, [])
                idx = bisect_right(lst, x)
                x = min(x + c, lst[idx] - 1) if idx < len(lst) else x + c
            else:                                        # tây: giảm x trên hàng y
                lst = rows.get(y, [])
                idx = bisect_left(lst, x) - 1
                x = max(x - c, lst[idx] + 1) if idx >= 0 else x - c
            ans = max(ans, x * x + y * y)
    return ans

for commands, obstacles, expected in examples:
    got = robot_bisect(commands, obstacles)
    print(commands, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[4, -1, 3] -> 25 OK
[4, -1, 4, -2, 4] -> 65 OK
[6, -1, -1, 6] -> 36 OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def robotSim(self, commands: List[int], obstacles: List[List[int]]) -> int:
        from bisect import bisect_left, bisect_right
        rows, cols = {}, {}
        for ox, oy in obstacles:
            rows.setdefault(oy, []).append(ox)
            cols.setdefault(ox, []).append(oy)
        for lst in rows.values():
            lst.sort()
        for lst in cols.values():
            lst.sort()
        x = y = k = ans = 0
        for c in commands:
            if c == -2:
                k = (k + 3) % 4
            elif c == -1:
                k = (k + 1) % 4
            else:
                if k == 0:
                    lst = cols.get(x, [])
                    idx = bisect_right(lst, y)
                    y = min(y + c, lst[idx] - 1) if idx < len(lst) else y + c
                elif k == 2:
                    lst = cols.get(x, [])
                    idx = bisect_left(lst, y) - 1
                    y = max(y - c, lst[idx] + 1) if idx >= 0 else y - c
                elif k == 1:
                    lst = rows.get(y, [])
                    idx = bisect_right(lst, x)
                    x = min(x + c, lst[idx] - 1) if idx < len(lst) else x + c
                else:
                    lst = rows.get(y, [])
                    idx = bisect_left(lst, x) - 1
                    x = max(x - c, lst[idx] + 1) if idx >= 0 else x - c
                ans = max(ans, x * x + y * y)
        return ans

sol_bisect = Solution().robotSim
assert [sol_bisect(c, o) for c, o, _ in examples] == [e for _, _, e in examples]
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** tiền xử lý `O(m log m)` (sắp xếp); mỗi lệnh `O(log m)`; tổng `O(m log m + n log m)`; bộ nhớ `O(m)`.

## 8. TRACE từng bước

### 8.1. Cách 2 cho ví dụ 2: `commands = [4,-1,4,-2,4]`, `obstacles = [[2,4]]`

In từng lệnh, từng bước đi, ô bị chặn, và `ans`.

In [13]:
def trace_set(commands, obstacles):
    dirs = (0, 1, 0, -1, 0)
    names = ["bắc", "đông", "nam", "tây"]
    s = {(x, y) for x, y in obstacles}
    x = y = k = ans = 0
    for c in commands:
        if c == -2:
            k = (k + 3) % 4
            print(f"lệnh -2: quay trái -> hướng {names[k]}")
        elif c == -1:
            k = (k + 1) % 4
            print(f"lệnh -1: quay phải -> hướng {names[k]}")
        else:
            print(f"lệnh {c}: đi {c} bước hướng {names[k]} từ ({x},{y})")
            for step in range(c):
                nx, ny = x + dirs[k], y + dirs[k + 1]
                if (nx, ny) in s:
                    print(f"    bước {step + 1}: ô ({nx},{ny}) là chướng ngại -> dừng ở ({x},{y})")
                    break
                x, y = nx, ny
                ans = max(ans, x * x + y * y)
                print(f"    bước {step + 1}: tới ({x},{y}), x^2+y^2={x * x + y * y}, ans={ans}")
    return ans

assert trace_set([4, -1, 4, -2, 4], [[2, 4]]) == 65

lệnh 4: đi 4 bước hướng bắc từ (0,0)
    bước 1: tới (0,1), x^2+y^2=1, ans=1
    bước 2: tới (0,2), x^2+y^2=4, ans=4
    bước 3: tới (0,3), x^2+y^2=9, ans=9
    bước 4: tới (0,4), x^2+y^2=16, ans=16
lệnh -1: quay phải -> hướng đông
lệnh 4: đi 4 bước hướng đông từ (0,4)
    bước 1: tới (1,4), x^2+y^2=17, ans=17
    bước 2: ô (2,4) là chướng ngại -> dừng ở (1,4)
lệnh -2: quay trái -> hướng bắc
lệnh 4: đi 4 bước hướng bắc từ (1,4)
    bước 1: tới (1,5), x^2+y^2=26, ans=26
    bước 2: tới (1,6), x^2+y^2=37, ans=37
    bước 3: tới (1,7), x^2+y^2=50, ans=50
    bước 4: tới (1,8), x^2+y^2=65, ans=65


### 8.2. Cách 2 cho ví dụ 3 (chướng ngại ở gốc)

`commands = [6,-1,-1,6]`, `obstacles = [[0,0]]`: robot đi bắc 6 bước tới `(0,6)`; hai lần quay phải đưa hướng về **nam**; đi nam: tới `(0,5), (0,4), (0,3), (0,2), (0,1)`, bước kế `(0,0)` là chướng ngại nên dừng ở `(0,1)`. Đáp án là `36` (tại `(0,6)`).

In [14]:
assert trace_set([6, -1, -1, 6], [[0, 0]]) == 36

lệnh 6: đi 6 bước hướng bắc từ (0,0)
    bước 1: tới (0,1), x^2+y^2=1, ans=1
    bước 2: tới (0,2), x^2+y^2=4, ans=4
    bước 3: tới (0,3), x^2+y^2=9, ans=9
    bước 4: tới (0,4), x^2+y^2=16, ans=16
    bước 5: tới (0,5), x^2+y^2=25, ans=25
    bước 6: tới (0,6), x^2+y^2=36, ans=36
lệnh -1: quay phải -> hướng đông
lệnh -1: quay phải -> hướng nam
lệnh 6: đi 6 bước hướng nam từ (0,6)
    bước 1: tới (0,5), x^2+y^2=25, ans=36
    bước 2: tới (0,4), x^2+y^2=16, ans=36
    bước 3: tới (0,3), x^2+y^2=9, ans=36
    bước 4: tới (0,2), x^2+y^2=4, ans=36
    bước 5: tới (0,1), x^2+y^2=1, ans=36
    bước 6: ô (0,0) là chướng ngại -> dừng ở (0,1)


### 8.3. Cách 4 (nhảy bằng `bisect`) cho ví dụ 2

`cols[2] = [4]` (chướng ngại `(2,4)`), `rows[4] = [2]`. Sau lệnh `4` (bắc): trên cột `x = 0` không có chướng ngại nên `y = 0 + 4 = 4`. Quay phải (đông). Lệnh `4` đi đông trên hàng `y = 4`: `rows[4] = [2]`, `bisect_right([2], 0) = 0` nên chướng ngại đầu tiên ở `x_o = 2`; `x = min(0 + 4, 2 - 1) = 1`. Quay trái (bắc). Lệnh `4`: cột `x = 1` không có chướng ngại: `y = 8`. Đáp án `1 + 64 = 65`.

In [15]:
def trace_bisect(commands, obstacles):
    rows, cols = defaultdict(list), defaultdict(list)
    for ox, oy in obstacles:
        rows[oy].append(ox)
        cols[ox].append(oy)
    print("rows =", dict(rows), "| cols =", dict(cols))
    x = y = k = ans = 0
    for c in commands:
        if c == -2:
            k = (k + 3) % 4
        elif c == -1:
            k = (k + 1) % 4
        else:
            old = (x, y)
            if k == 0:
                lst = cols.get(x, []); idx = bisect_right(lst, y)
                y = min(y + c, lst[idx] - 1) if idx < len(lst) else y + c
            elif k == 2:
                lst = cols.get(x, []); idx = bisect_left(lst, y) - 1
                y = max(y - c, lst[idx] + 1) if idx >= 0 else y - c
            elif k == 1:
                lst = rows.get(y, []); idx = bisect_right(lst, x)
                x = min(x + c, lst[idx] - 1) if idx < len(lst) else x + c
            else:
                lst = rows.get(y, []); idx = bisect_left(lst, x) - 1
                x = max(x - c, lst[idx] + 1) if idx >= 0 else x - c
            ans = max(ans, x * x + y * y)
            print(f"lệnh {c} hướng {['bắc', 'đông', 'nam', 'tây'][k]}: {old} -> ({x},{y}), ans={ans}")
    return ans

assert trace_bisect([4, -1, 4, -2, 4], [[2, 4]]) == 65

rows = {4: [2]} | cols = {2: [4]}
lệnh 4 hướng bắc: (0, 0) -> (0,4), ans=16
lệnh 4 hướng đông: (0, 4) -> (1,4), ans=17
lệnh 4 hướng bắc: (1, 4) -> (1,8), ans=65


## 9. Code nộp LeetCode

Ta chọn **Cách 2** (`set` các tuple + mảng hướng): ngắn, dễ đúng, `O(C * n + m)`; Cách 4 chỉ đáng dùng nếu số bước mỗi lệnh lớn.

```python
class Solution:
    def robotSim(self, commands: List[int], obstacles: List[List[int]]) -> int:
        dirs = (0, 1, 0, -1, 0)
        s = {(x, y) for x, y in obstacles}
        ans = k = 0
        x = y = 0
        for c in commands:
            if c == -2:
                k = (k + 3) % 4
            elif c == -1:
                k = (k + 1) % 4
            else:
                for _ in range(c):
                    nx, ny = x + dirs[k], y + dirs[k + 1]
                    if (nx, ny) in s:
                        break
                    x, y = nx, ny
                    ans = max(ans, x * x + y * y)
        return ans
```

In [16]:
class Solution:
    def robotSim(self, commands: List[int], obstacles: List[List[int]]) -> int:
        dirs = (0, 1, 0, -1, 0)
        s = {(x, y) for x, y in obstacles}
        ans = k = 0
        x = y = 0
        for c in commands:
            if c == -2:
                k = (k + 3) % 4
            elif c == -1:
                k = (k + 1) % 4
            else:
                for _ in range(c):
                    nx, ny = x + dirs[k], y + dirs[k + 1]
                    if (nx, ny) in s:
                        break
                    x, y = nx, ny
                    ans = max(ans, x * x + y * y)
        return ans

sol = Solution()
for commands, obstacles, expected in examples:
    got = sol.robotSim(commands, obstacles)
    print(commands, obstacles, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[4, -1, 3] [] -> 25 OK
[4, -1, 4, -2, 4] [[2, 4]] -> 65 OK
[6, -1, -1, 6] [[0, 0]] -> 36 OK


## 10. Vì sao đúng

**(a) Trạng thái đầy đủ.** Hành vi tương lai của robot chỉ phụ thuộc vào **vị trí hiện tại** `(x, y)` và **hướng** `k`: quay chỉ đổi `k`; đi tiến cập nhật `(x, y)` theo `(dirs[k], dirs[k + 1])` và dừng khi ô kế nằm trong tập chướng ngại. Đó đúng là luật của đề, nên mô phỏng từng lệnh tuần tự cho ra đúng đường đi.

**(b) Phép quay.** Hướng theo thứ tự bắc, đông, nam, tây là thứ tự quay **phải**: quay phải cộng `1`, quay trái cộng `3` (tương đương trừ `1`) theo modulo `4`.

**(c) Khoảng cách tối đa.** Đáp án là cực đại của `x² + y²` trên **mọi** vị trí robot từng đứng. Mỗi lần vị trí đổi (chỉ ở lệnh đi, từng bước thành công), ta cập nhật `ans`; vị trí xuất phát `(0, 0)` cho `0` (là giá trị khởi tạo). Do đó `ans` đúng là cực đại cần tìm.

**(d) Chướng ngại tại `(0, 0)`.** Robot không kiểm tra chướng ngại ở **ô đang đứng**, chỉ kiểm tra **ô kế tiếp**, nên chướng ngại ở gốc bị bỏ qua lúc xuất phát; còn khi robot đi từ `(0,1)` xuống thì ô kế tiếp là `(0, 0)` nằm trong tập chướng ngại nên bị chặn, đúng như đề ("không thể quay lại `(0,0)`").

**(e) Cách 4.** Với hướng đông trên hàng `y`, ô sát chướng ngại gần nhất phía đông là `x_o - 1` (với `x_o` là chướng ngại nhỏ nhất `> x`); đi `c` bước kết thúc ở `min(x + c, x_o - 1)`. `bisect_right` cho đúng "nhỏ nhất `> x`". Tương tự cho các hướng còn lại với `bisect_left(...) - 1` là "lớn nhất `< x`". Vị trí cuối lệnh là vị trí duy nhất cần xét vì `x² + y²` lồi dọc đoạn thẳng (mục 3.4) và đầu mút đầu đã được xét ở lệnh trước.

**Kiểm chứng bằng máy.** Oracle **độc lập**: mô phỏng bằng **chữ cái** hướng `N/E/S/W` (bảng chuyển hướng viết tay, không dùng `% 4` hay `dirs`), khóa ô dạng **chuỗi** `"x,y"`, và ghi lại **toàn bộ đường đi** rồi lấy `max` bình phương khoảng cách. So sánh với mọi cách trên ví dụ, edge case và dữ liệu ngẫu nhiên (lưới nhỏ để va chạm nhiều, lưới lớn).

In [17]:
def oracle(commands: List[int], obstacles: List[List[int]]) -> int:
    left = {"N": "W", "W": "S", "S": "E", "E": "N"}
    right = {"N": "E", "E": "S", "S": "W", "W": "N"}
    delta = {"N": (0, 1), "E": (1, 0), "S": (0, -1), "W": (-1, 0)}
    blocked = {f"{x},{y}" for x, y in obstacles}
    heading, px, py = "N", 0, 0
    path = [(0, 0)]
    for c in commands:
        if c == -2:
            heading = left[heading]
        elif c == -1:
            heading = right[heading]
        else:
            for _ in range(c):
                nx, ny = px + delta[heading][0], py + delta[heading][1]
                if f"{nx},{ny}" in blocked:
                    break
                px, py = nx, ny
                path.append((px, py))
    return max(a * a + b * b for a, b in path)

for commands, obstacles, expected in examples:
    assert oracle(commands, obstacles) == expected
print("oracle (chữ N/E/S/W, khóa chuỗi, ghi cả đường đi) khớp ví dụ")

oracle (chữ N/E/S/W, khóa chuỗi, ghi cả đường đi) khớp ví dụ


## 11. Time / Space Complexity

Gọi `n = len(commands)`, `m = len(obstacles)`, `C = 9` (số bước tối đa mỗi lệnh).

| Cách | Time | Space |
|---|---|---|
| 1. Quét danh sách | `O(C * n * m)` | `O(m)` |
| 2. `set` tuple | `O(C * n + m)` | `O(m)` |
| 3. `set` số phức | `O(C * n + m)` | `O(m)` |
| 4. Hàng/cột + `bisect` | `O(m log m + n log m)` | `O(m)` |

Đo thử với `n = 10^4` lệnh và `m = 10^4` chướng ngại (cỡ tối đa):

In [18]:
rng = random.Random(11)
n_cmds, m = 10000, 10000
commands = [rng.choice([-2, -1] + list(range(1, 10))) for _ in range(n_cmds)]
obstacles = [[rng.randint(-300, 300), rng.randint(-300, 300)] for _ in range(m)]
ref = oracle(commands, obstacles)
for name, fn in (("Cách 2 set tuple", sol_set), ("Cách 3 số phức", sol_complex), ("Cách 4 bisect", sol_bisect), ("nộp", Solution().robotSim)):
    t0 = time.perf_counter()
    got = fn(commands, obstacles)
    dt = (time.perf_counter() - t0) * 1000
    assert got == ref, name
    print(f"{name:18} {dt:8.1f} ms (đáp án {got})")
t0 = time.perf_counter()
robot_brute(commands[:1500], obstacles)
print(f"Cách 1 quét danh sách, chỉ 1500 lệnh đầu: {1000 * (time.perf_counter() - t0):.0f} ms")

Cách 2 set tuple       16.2 ms (đáp án 1562881)
Cách 3 số phức         23.5 ms (đáp án 1562881)
Cách 4 bisect           7.9 ms (đáp án 1562881)


nộp                    17.1 ms (đáp án 1562881)


Cách 1 quét danh sách, chỉ 1500 lệnh đầu: 676 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với oracle, và test ngẫu nhiên có seed (đúng ràng buộc của đề).

In [19]:
def solutions_list():
    return {
        "set": robot_set,
        "complex": robot_complex,
        "bisect": robot_bisect,
        "sol_set": sol_set,
        "sol_complex": sol_complex,
        "sol_bisect": sol_bisect,
        "final": Solution().robotSim,
    }

def run_all(commands, obstacles, expected=None, include_brute=True):
    ref = oracle(commands, obstacles)
    for name, fn in solutions_list().items():
        assert fn(list(commands), [list(o) for o in obstacles]) == ref, (name, commands, obstacles)
    if include_brute:
        assert robot_brute(commands, obstacles) == ref, ("brute", commands, obstacles)
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", commands, obstacles, expected, ref)
    return ref

# 12.1 ví dụ đề
for commands, obstacles, expected in examples:
    assert run_all(commands, obstacles, expected) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([-1, -1, -2, 5], [], 25),                   # phải, phải, trái -> hướng đông; đi 5: x = 5
    ([-2], [], 0),                               # chỉ quay: không đi đâu
    ([1], [], 1),                                # một bước bắc
    ([1], [[0, 1]], 0),                          # chướng ngại ngay trước mặt
    ([9], [[0, 9]], 64),                         # chặn ở bước 9: dừng ở (0,8)
    ([5, -2, 3, -2, 2], [[-3, 5]], 29),          # (−2,5): 4+25=29
    ([4, -1, -1, 4], [], 16),                    # đi ra rồi quay về gốc: cực đại không ở cuối
    ([4, -1, 4, -1, 4, -1, 4], [], 32),          # đi vuông (4,4) rồi về: max 4^2+4^2
    ([3, -1, 3, -1, 3, -1, 3], [], 18),          # (3,3)
    ([6, -1, -1, 6], [[0, 0]], 36),              # ví dụ 3: chướng ngại ở gốc
    ([1, -1, -1, 1], [[0, 0]], 1),               # (0,1) rồi chặn vì (0,0) là chướng ngại
    ([1, -1, -1, 1, 1], [], 1),                  # sau khi về gốc đi tiếp 1 bước nam: (0,-1) -> 1
    ([5, -1, 5, -1, 5, -1, 5], [[0, 0]], 50),    # đi vuông cạnh 5, ô cuối (0,0) bị chặn
    ([9] * 5000, [], 45000 ** 2),                # chạy thẳng 45000 bước: 2.025e9 < 2^31
    ([9] * 5000, [[0, 20000]], 19999 ** 2),      # bị chặn ở (0,19999)
    ([-2, 9, -2, 9], [[-9, 0]], 145),            # tây bị chặn ở (-8,0), rồi trái = nam 9 bước: (-8,-9): 64 + 81
]
for commands, obstacles, expected in edge:
    assert run_all(commands, obstacles, expected, include_brute=len(commands) <= 200) == expected, (commands, obstacles)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 16 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_jump_no_check(commands, obstacles):            # đi liền c ô, chỉ kiểm tra ô đích
    dirs = (0, 1, 0, -1, 0)
    s = {(x, y) for x, y in obstacles}
    x = y = k = ans = 0
    for c in commands:
        if c == -2:
            k = (k + 3) % 4
        elif c == -1:
            k = (k + 1) % 4
        else:
            nx, ny = x + c * dirs[k], y + c * dirs[k + 1]
            if (nx, ny) not in s:
                x, y = nx, ny
                ans = max(ans, x * x + y * y)
    return ans

def wrong_only_final(commands, obstacles):               # chỉ lấy khoảng cách ở cuối đường đi
    dirs = (0, 1, 0, -1, 0)
    s = {(x, y) for x, y in obstacles}
    x = y = k = 0
    for c in commands:
        if c == -2:
            k = (k + 3) % 4
        elif c == -1:
            k = (k + 1) % 4
        else:
            for _ in range(c):
                nx, ny = x + dirs[k], y + dirs[k + 1]
                if (nx, ny) in s:
                    break
                x, y = nx, ny
    return x * x + y * y

def wrong_swap_turn(commands, obstacles):                # đảo quay trái / quay phải
    dirs = (0, 1, 0, -1, 0)
    s = {(x, y) for x, y in obstacles}
    x = y = k = ans = 0
    for c in commands:
        if c == -1:
            k = (k + 3) % 4
        elif c == -2:
            k = (k + 1) % 4
        else:
            for _ in range(c):
                nx, ny = x + dirs[k], y + dirs[k + 1]
                if (nx, ny) in s:
                    break
                x, y = nx, ny
                ans = max(ans, x * x + y * y)
    return ans

def wrong_move_then_check(commands, obstacles):          # đi vào ô rồi mới kiểm tra (đứng trên chướng ngại)
    dirs = (0, 1, 0, -1, 0)
    s = {(x, y) for x, y in obstacles}
    x = y = k = ans = 0
    for c in commands:
        if c == -2:
            k = (k + 3) % 4
        elif c == -1:
            k = (k + 1) % 4
        else:
            for _ in range(c):
                x, y = x + dirs[k], y + dirs[k + 1]
                if (x, y) in s:
                    break
                ans = max(ans, x * x + y * y)
    return ans

def wrong_origin_blocks(commands, obstacles):            # coi chướng ngại ở (0,0) là chặn ngay từ đầu
    if [0, 0] in [list(o) for o in obstacles]:
        return 0
    return Solution().robotSim(commands, obstacles)

def find_counter(wrong, tries=20000, seed=1):
    rng = random.Random(seed)
    for _ in range(tries):
        commands = [rng.choice([-2, -1, 1, 2, 3, 4, 5, 6]) for _ in range(rng.randint(1, 7))]
        obstacles = [[rng.randint(-4, 4), rng.randint(-4, 4)] for _ in range(rng.randint(0, 4))]
        if wrong(commands, obstacles) != oracle(commands, obstacles):
            return commands, obstacles
    return None

counters = {}
for name, fn in (("đi liền c ô, chỉ kiểm tra ô đích", wrong_jump_no_check), ("chỉ lấy khoảng cách ở cuối", wrong_only_final),
                 ("đảo quay trái/phải", wrong_swap_turn), ("đi vào ô rồi mới kiểm tra", wrong_move_then_check),
                 ("chướng ngại ở gốc chặn từ đầu", wrong_origin_blocks)):
    c = find_counter(fn)
    assert c is not None, name
    counters[name] = c
    commands, obstacles = c
    print(f"{name:36} commands={commands} obstacles={obstacles}: sai = {fn(commands, obstacles)}, đúng = {oracle(commands, obstacles)}")
    assert run_all(commands, obstacles) == oracle(commands, obstacles)
print("12.3 phản ví dụ: OK")

đi liền c ô, chỉ kiểm tra ô đích     commands=[-2, 5] obstacles=[[1, 2], [-4, 0], [-2, -1], [-4, 0]]: sai = 25, đúng = 9
chỉ lấy khoảng cách ở cuối           commands=[3, 2, 4, -1, -1, 2, 2] obstacles=[]: sai = 25, đúng = 81
đảo quay trái/phải                   commands=[-1, 1, 5, 4, 6, -2] obstacles=[[-4, 0], [2, -2], [-2, 4]]: sai = 9, đúng = 256
đi vào ô rồi mới kiểm tra            commands=[1, 1] obstacles=[[-1, 0], [1, 4], [0, 1], [1, 1]]: sai = 4, đúng = 0
chướng ngại ở gốc chặn từ đầu        commands=[-2, -2, 4, 1, 1] obstacles=[[0, 0]]: sai = 0, đúng = 36
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên lưới nhỏ (nhiều va chạm, chướng ngại có thể ở gốc, trùng nhau)
rng = random.Random(4874)
for _ in range(2500):
    commands = [rng.choice([-2, -1] + list(range(1, 10))) for _ in range(rng.randint(1, 25))]
    side = rng.choice([2, 4, 8, 15])
    obstacles = [[rng.randint(-side, side), rng.randint(-side, side)] for _ in range(rng.randint(0, 25))]
    run_all(commands, obstacles)
print("12.4 ngẫu nhiên lưới nhỏ (2500 ca, gồm brute): OK")

# 12.5 ngẫu nhiên trung bình: toạ độ rộng, nhiều chướng ngại đặt trên đường đi để chắc chắn có va chạm
rng = random.Random(5874)
for _ in range(120):
    commands = [rng.choice([-2, -1] + list(range(1, 10))) for _ in range(rng.randint(100, 600))]
    # chạy một lần mô phỏng để lấy các ô đã đi qua rồi đặt chướng ngại ngay trên đường đó
    dirs = (0, 1, 0, -1, 0)
    x = y = k = 0
    visited = []
    for c in commands:
        if c == -2:
            k = (k + 3) % 4
        elif c == -1:
            k = (k + 1) % 4
        else:
            for _ in range(c):
                x, y = x + dirs[k], y + dirs[k + 1]
                visited.append([x, y])
    obstacles = rng.sample(visited, min(len(visited), rng.randint(0, 60)))
    obstacles += [[rng.randint(-30000, 30000), rng.randint(-30000, 30000)] for _ in range(rng.randint(0, 50))]
    run_all(commands, obstacles, include_brute=False)
print("12.5 chướng ngại đặt trên đường đi (120 ca): OK")

# 12.6 lớn: n = 10^4 lệnh, m = 10^4 chướng ngại, nhiều dạng
rng = random.Random(6874)
shapes = {}
shapes["ngẫu nhiên"] = ([rng.choice([-2, -1] + list(range(1, 10))) for _ in range(10000)],
                         [[rng.randint(-300, 300), rng.randint(-300, 300)] for _ in range(10000)])
shapes["không chướng ngại"] = ([rng.choice([-2, -1] + list(range(1, 10))) for _ in range(10000)], [])
shapes["chạy thẳng bắc"] = ([9] * 5000, [])
shapes["đi vuông lặp"] = ([9, -1] * 5000, [[0, 0]])
shapes["biên toạ độ"] = ([rng.choice([-2, -1, 9, 9, 9]) for _ in range(10000)],
                         [[rng.choice([-30000, 30000]), rng.randint(-30000, 30000)] for _ in range(10000)])
for name, (commands, obstacles) in shapes.items():
    ref = oracle(commands, obstacles)
    assert ref < 2 ** 31, name
    for fname, fn in solutions_list().items():
        assert fn(list(commands), [list(o) for o in obstacles]) == ref, (name, fname)
    print(f"  {name:20} đáp án {ref}")
print("12.6 n = m = 10^4: OK")

# 12.7 tính chất đối xứng gương: lấy đối xứng qua trục Y (x -> -x) và hoán đổi quay trái/phải thì đáp án không đổi
rng = random.Random(7874)
swap = {-1: -2, -2: -1}
for _ in range(500):
    commands = [rng.choice([-2, -1] + list(range(1, 10))) for _ in range(rng.randint(1, 40))]
    obstacles = [[rng.randint(-6, 6), rng.randint(-6, 6)] for _ in range(rng.randint(0, 12))]
    mirrored_cmds = [swap.get(c, c) for c in commands]
    mirrored_obs = [[-x, y] for x, y in obstacles]
    assert Solution().robotSim(commands, obstacles) == Solution().robotSim(mirrored_cmds, mirrored_obs)
print("12.7 tính chất đối xứng gương: OK")

12.4 ngẫu nhiên lưới nhỏ (2500 ca, gồm brute): OK


12.5 chướng ngại đặt trên đường đi (120 ca): OK
  ngẫu nhiên           đáp án 825098
  không chướng ngại    đáp án 2530802


  chạy thẳng bắc       đáp án 2025000000
  đi vuông lặp         đáp án 181


  biên toạ độ          đáp án 8444817
12.6 n = m = 10^4: OK
12.7 tính chất đối xứng gương: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Đi liền `c` ô, chỉ kiểm tra ô đích

Chướng ngại có thể nằm **giữa đường**, không chỉ ở ô đích.

In [22]:
commands, obstacles = counters["đi liền c ô, chỉ kiểm tra ô đích"]
print("commands =", commands, "obstacles =", obstacles, "| sai ->", wrong_jump_no_check(commands, obstacles), "| đúng ->", Solution().robotSim(commands, obstacles))
assert wrong_jump_no_check(commands, obstacles) != Solution().robotSim(commands, obstacles)
print("ví dụ 2:", wrong_jump_no_check([4, -1, 4, -2, 4], [[2, 4]]), "(sai, đi xuyên tới (4,4)) so với", Solution().robotSim([4, -1, 4, -2, 4], [[2, 4]]))

commands = [-2, 5] obstacles = [[1, 2], [-4, 0], [-2, -1], [-4, 0]] | sai -> 25 | đúng -> 9
ví dụ 2: 80 (sai, đi xuyên tới (4,4)) so với 65


Vì sao sai: đề nói robot đi **từng đơn vị một** và bị chặn ngay khi ô kế là chướng ngại. **Đúng:** lặp từng bước (hoặc dùng `bisect` tìm chướng ngại gần nhất như Cách 4).

### Lỗi 2 — Chỉ lấy khoảng cách ở **cuối** đường đi

Đề hỏi khoảng cách lớn nhất **tại mọi điểm trên đường đi**, không phải khoảng cách cuối.

In [23]:
commands, obstacles = counters["chỉ lấy khoảng cách ở cuối"]
print("commands =", commands, "obstacles =", obstacles, "| sai ->", wrong_only_final(commands, obstacles), "| đúng ->", Solution().robotSim(commands, obstacles))
assert wrong_only_final(commands, obstacles) < Solution().robotSim(commands, obstacles)
print("ví dụ gọn: [4, -1, -1, 4] ->", wrong_only_final([4, -1, -1, 4], []), "(sai) so với", Solution().robotSim([4, -1, -1, 4], []))

commands = [3, 2, 4, -1, -1, 2, 2] obstacles = [] | sai -> 25 | đúng -> 81
ví dụ gọn: [4, -1, -1, 4] -> 0 (sai) so với 16


Vì sao sai: robot có thể đi xa rồi quay về gốc. **Đúng:** cập nhật `ans = max(ans, x*x + y*y)` sau **mỗi** bước thành công.

### Lỗi 3 — Đảo quay trái / quay phải

`-2` là quay **trái**, `-1` là quay **phải** (ngược với thứ tự số thường gặp). Hướng theo thứ tự bắc, đông, nam, tây là chiều quay phải.

In [24]:
commands, obstacles = counters["đảo quay trái/phải"]
print("commands =", commands, "obstacles =", obstacles, "| sai ->", wrong_swap_turn(commands, obstacles), "| đúng ->", Solution().robotSim(commands, obstacles))
assert wrong_swap_turn(commands, obstacles) != Solution().robotSim(commands, obstacles)

commands = [-1, 1, 5, 4, 6, -2] obstacles = [[-4, 0], [2, -2], [-2, 4]] | sai -> 9 | đúng -> 256


Vì sao sai: không có chướng ngại thì đối xứng trái/phải che mất lỗi (khoảng cách không đổi), nhưng chướng ngại phá đối xứng. **Đúng:** `-1 -> (k + 1) % 4`, `-2 -> (k + 3) % 4`.

### Lỗi 4 — Di chuyển trước rồi mới kiểm tra chướng ngại

Nếu cập nhật `(x, y)` rồi mới kiểm tra, robot **đứng trên** chướng ngại thay vì dừng ở ô liền kề.

In [25]:
commands, obstacles = counters["đi vào ô rồi mới kiểm tra"]
print("commands =", commands, "obstacles =", obstacles, "| sai ->", wrong_move_then_check(commands, obstacles), "| đúng ->", Solution().robotSim(commands, obstacles))
assert wrong_move_then_check(commands, obstacles) != Solution().robotSim(commands, obstacles)

commands = [1, 1] obstacles = [[-1, 0], [1, 4], [0, 1], [1, 1]] | sai -> 4 | đúng -> 0


Vì sao sai: vị trí sau lệnh sai (đứng trong ô chướng ngại) nên các lệnh sau đi lệch và tính sai khoảng cách. **Đúng:** tính ô kế `(nx, ny)`, kiểm tra, rồi mới gán `x, y = nx, ny`.

### Lỗi 5 — Coi chướng ngại ở `(0, 0)` chặn robot từ đầu

Đề nói robot **bỏ qua** chướng ngại ở gốc cho tới khi rời gốc.

In [26]:
commands, obstacles = counters["chướng ngại ở gốc chặn từ đầu"]
print("commands =", commands, "obstacles =", obstacles, "| sai ->", wrong_origin_blocks(commands, obstacles), "| đúng ->", Solution().robotSim(commands, obstacles))
assert wrong_origin_blocks(commands, obstacles) != Solution().robotSim(commands, obstacles)
print("ví dụ 3:", wrong_origin_blocks([6, -1, -1, 6], [[0, 0]]), "(sai) so với", Solution().robotSim([6, -1, -1, 6], [[0, 0]]))

commands = [-2, -2, 4, 1, 1] obstacles = [[0, 0]] | sai -> 0 | đúng -> 36
ví dụ 3: 0 (sai) so với 36


Vì sao sai: robot không "đứng trong" chướng ngại; chỉ cần ô **kế tiếp** không phải chướng ngại là đi được. Kiểm tra ô kế (không phải ô hiện tại) tự xử lý đúng trường hợp này.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "robot / con trỏ đi theo lệnh trên lưới, quay trái phải, có chướng ngại" ⇒ **mô phỏng** với trạng thái `(x, y, hướng)`.
- **Mảng hướng:** `dirs = (0, 1, 0, -1, 0)` và `(dirs[k], dirs[k + 1])`; quay bằng `% 4` (bắc, đông, nam, tây là chiều quay phải).
- **Chướng ngại:** luôn đưa vào `set` các tuple (hoặc số phức / khóa số nguyên `x * W + y`); đừng quét danh sách.
- **Kiểm tra ô kế trước khi đi**, không sau.
- **Cực đại khoảng cách:** cập nhật sau mỗi bước (hoặc ở đầu mút khi nhảy).
- Cùng họ: Robot Bounded In Circle (1041), Walking Robot Simulation II (2069), Spiral Matrix (54), Robot Room Cleaner (489), Where Will the Ball Fall (1706).
- Test nên có: chướng ngại ngay trước mặt, chướng ngại ở `(0, 0)`, đi ra rồi quay về, chạy thẳng 45000 bước, toạ độ biên `±3 * 10^4`.

## 15. Biến thể

1. **Số ô khác nhau đã đi qua** và số lệnh đi bị chặn (bài tập 5).
2. **Khoảng cách Manhattan lớn nhất** hoặc **Chebyshev**: chỉ đổi biểu thức khoảng cách.
3. **Số bước mỗi lệnh lớn** (ví dụ `10^9`): Cách 4 (nhảy bằng `bisect`) giữ `O(log m)` mỗi lệnh.
4. Bài liên quan: Robot Bounded In Circle (1041).

Dưới đây là biến thể 2 (Manhattan) kiểm bằng mô phỏng độc lập.

In [27]:
def robot_manhattan(commands: List[int], obstacles: List[List[int]]) -> int:
    dirs = (0, 1, 0, -1, 0)
    s = {(x, y) for x, y in obstacles}
    x = y = k = ans = 0
    for c in commands:
        if c == -2:
            k = (k + 3) % 4
        elif c == -1:
            k = (k + 1) % 4
        else:
            for _ in range(c):
                nx, ny = x + dirs[k], y + dirs[k + 1]
                if (nx, ny) in s:
                    break
                x, y = nx, ny
                ans = max(ans, abs(x) + abs(y))
    return ans

def manhattan_oracle(commands, obstacles):
    left = {"N": "W", "W": "S", "S": "E", "E": "N"}
    right = {"N": "E", "E": "S", "S": "W", "W": "N"}
    delta = {"N": (0, 1), "E": (1, 0), "S": (0, -1), "W": (-1, 0)}
    blocked = {f"{a},{b}" for a, b in obstacles}
    h, px, py, best = "N", 0, 0, 0
    for c in commands:
        if c == -2:
            h = left[h]
        elif c == -1:
            h = right[h]
        else:
            for _ in range(c):
                nx, ny = px + delta[h][0], py + delta[h][1]
                if f"{nx},{ny}" in blocked:
                    break
                px, py = nx, ny
                best = max(best, abs(px) + abs(py))
    return best

rng = random.Random(15)
for _ in range(500):
    commands = [rng.choice([-2, -1] + list(range(1, 10))) for _ in range(rng.randint(1, 20))]
    obstacles = [[rng.randint(-5, 5), rng.randint(-5, 5)] for _ in range(rng.randint(0, 10))]
    assert robot_manhattan(commands, obstacles) == manhattan_oracle(commands, obstacles)
print([robot_manhattan(c, o) for c, o, _ in examples], "(Manhattan cho 3 ví dụ: 7, 9, 6)")
assert [robot_manhattan(c, o) for c, o, _ in examples] == [7, 9, 6]

[7, 9, 6] (Manhattan cho 3 ví dụ: 7, 9, 6)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét danh sách | Mô phỏng + `in list` | `O(C n m)` | `O(m)` | Oracle nhỏ |
| 2. `set` tuple | Mô phỏng + bảng băm | `O(C n + m)` | `O(m)` | **Cách nộp chính** |
| 3. Số phức | Vectơ hướng bằng `1j` | `O(C n + m)` | `O(m)` | Gọn, độc đáo |
| 4. Hàng/cột + `bisect` | Nhảy tới chướng ngại | `O((n + m) log m)` | `O(m)` | Số bước mỗi lệnh lớn |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
dirs = (0, 1, 0, -1, 0)
k = (0 + 3) % 4
print(k, (dirs[k], dirs[k + 1]))
print((3 + 1) % 4)
print(1j * 1j)
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `commands = [5, -2, 3, -2, 2]`, `obstacles = [[-3, 5]]`: ghi vị trí cuối của robot và đáp án (bình phương khoảng cách lớn nhất).

In [29]:
# Bài 2: điền kết quả tính tay
hand_final_position = None      # (x, y)
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành mô phỏng bằng `set` và mảng hướng.

In [30]:
def exercise_3_simulate(commands: List[int], obstacles: List[List[int]]) -> int:
    dirs = (0, 1, 0, -1, 0)
    s = {(x, y) for x, y in obstacles}
    x = y = k = ans = 0
    for c in commands:
        if c == -2:
            pass    # TODO 1: quay trái
        elif c == -1:
            pass    # TODO 2: quay phải
        else:
            for _ in range(c):
                # TODO 3: tính ô kế (nx, ny); nếu là chướng ngại thì break;
                #         ngược lại cập nhật x, y và ans
                pass
    return ans


# for commands, obstacles, expected in examples:
#     assert exercise_3_simulate(commands, obstacles) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 3, hãy tự viết `exercise_4_complex(commands, obstacles)` dùng số phức cho vị trí và hướng.

In [31]:
def exercise_4_complex(commands: List[int], obstacles: List[List[int]]) -> int:
    # TODO
    pass


# for commands, obstacles, expected in examples:
#     assert exercise_4_complex(commands, obstacles) == expected

### Bài 5 — Mini challenge: số ô đã đi qua và số lệnh bị chặn (khó)

Viết `exercise_5_stats(commands, obstacles)` trả về cặp `(visited, blocked)`:

- `visited`: số **ô khác nhau** robot từng đứng (kể cả `(0, 0)`);
- `blocked`: số **lệnh đi** mà robot bị chặn bởi chướng ngại ít nhất một lần (kể cả khi bị chặn ngay bước đầu).

```python
exercise_5_stats([4, -1, 3], [])                  -> (8, 0)
exercise_5_stats([4, -1, 4, -2, 4], [[2, 4]])      -> (10, 1)
exercise_5_stats([6, -1, -1, 6], [[0, 0]])         -> (7, 1)
```

In [32]:
def exercise_5_stats(commands: List[int], obstacles: List[List[int]]):
    # TODO
    pass


# assert exercise_5_stats([4, -1, 3], []) == (8, 0)
# assert exercise_5_stats([4, -1, 4, -2, 4], [[2, 4]]) == (10, 1)
# assert exercise_5_stats([6, -1, -1, 6], [[0, 0]]) == (7, 1)

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `(0 + 3) % 4` là chỉ số hướng sau khi quay trái từ bắc.
- Mức 2: hướng `3` là tây, vectơ `(dirs[3], dirs[4]) = (-1, 0)`; `1j * 1j = j^2`.
- Mức 3: `3 (-1, 0)`, `0`, `(-1+0j)`.

**Bài 2**
- Mức 1: đi bắc 5 tới `(0,5)`, quay trái hướng tây.
- Mức 2: tây 3 bước: `(-1,5)`, `(-2,5)`, bước kế `(-3,5)` là chướng ngại nên dừng ở `(-2,5)`; quay trái hướng nam, đi 2: `(-2,4)`, `(-2,3)`.
- Mức 3: vị trí cuối `(-2, 3)`; khoảng cách lớn nhất tại `(-2, 5)`: `4 + 25 = 29`.

**Bài 3**
- Mức 1: quay trái `k = (k + 3) % 4`, quay phải `k = (k + 1) % 4`.
- Mức 2: ô kế `nx, ny = x + dirs[k], y + dirs[k + 1]`.
- Mức 3: `if (nx, ny) in s: break`; `x, y = nx, ny`; `ans = max(ans, x * x + y * y)`.

**Bài 4**
- Mức 1: hướng ban đầu `d = 1j`, vị trí `pos = 0j`.
- Mức 2: quay trái `d *= 1j`, quay phải `d *= -1j`; ô kế `pos + d`.
- Mức 3: chướng ngại `{complex(x, y)}`; `ans = max(ans, int(pos.real) ** 2 + int(pos.imag) ** 2)`.

**Bài 5**
- Mức 1: dùng một `set` `visited = {(0, 0)}` và một bộ đếm `blocked`.
- Mức 2: trong lệnh đi, nếu gặp chướng ngại thì `blocked += 1` rồi `break`.
- Mức 3: thêm mỗi ô mới vào `visited` sau khi đi; trả `(len(visited), blocked)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
dirs = (0, 1, 0, -1, 0)
k = (0 + 3) % 4
print(k, (dirs[k], dirs[k + 1]))
print((3 + 1) % 4)
print(1j * 1j)

assert (k, (dirs[k], dirs[k + 1])) == (3, (-1, 0))
assert (3 + 1) % 4 == 0
assert 1j * 1j == (-1 + 0j)

3 (-1, 0)
0
(-1+0j)


### Đáp án Bài 2

Vị trí cuối `(-2, 3)`, đáp án `29`.

In [34]:
commands, obstacles = [5, -2, 3, -2, 2], [[-3, 5]]
hand_final_position_solution = (-2, 3)
hand_answer_solution = 29
assert Solution().robotSim(commands, obstacles) == hand_answer_solution == oracle(commands, obstacles)
# xác nhận vị trí cuối bằng mô phỏng riêng
dirs = (0, 1, 0, -1, 0)
s = {(a, b) for a, b in obstacles}
x = y = k = 0
for c in commands:
    if c == -2:
        k = (k + 3) % 4
    elif c == -1:
        k = (k + 1) % 4
    else:
        for _ in range(c):
            nx, ny = x + dirs[k], y + dirs[k + 1]
            if (nx, ny) in s:
                break
            x, y = nx, ny
assert (x, y) == hand_final_position_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_simulate_solution(commands: List[int], obstacles: List[List[int]]) -> int:
    dirs = (0, 1, 0, -1, 0)
    s = {(x, y) for x, y in obstacles}
    x = y = k = ans = 0
    for c in commands:
        if c == -2:
            k = (k + 3) % 4
        elif c == -1:
            k = (k + 1) % 4
        else:
            for _ in range(c):
                nx, ny = x + dirs[k], y + dirs[k + 1]
                if (nx, ny) in s:
                    break
                x, y = nx, ny
                ans = max(ans, x * x + y * y)
    return ans

for commands, obstacles, expected in examples:
    assert exercise_3_simulate_solution(commands, obstacles) == expected
rng = random.Random(3)
for _ in range(300):
    commands = [rng.choice([-2, -1] + list(range(1, 10))) for _ in range(rng.randint(1, 20))]
    obstacles = [[rng.randint(-5, 5), rng.randint(-5, 5)] for _ in range(rng.randint(0, 10))]
    assert exercise_3_simulate_solution(commands, obstacles) == oracle(commands, obstacles)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_complex_solution(commands: List[int], obstacles: List[List[int]]) -> int:
    s = {complex(x, y) for x, y in obstacles}
    pos, d = 0j, 1j
    ans = 0
    for c in commands:
        if c == -2:
            d *= 1j
        elif c == -1:
            d *= -1j
        else:
            for _ in range(c):
                nxt = pos + d
                if nxt in s:
                    break
                pos = nxt
                ans = max(ans, int(pos.real) ** 2 + int(pos.imag) ** 2)
    return ans

for commands, obstacles, expected in examples:
    assert exercise_4_complex_solution(commands, obstacles) == expected
rng = random.Random(4)
for _ in range(300):
    commands = [rng.choice([-2, -1] + list(range(1, 10))) for _ in range(rng.randint(1, 20))]
    obstacles = [[rng.randint(-5, 5), rng.randint(-5, 5)] for _ in range(rng.randint(0, 10))]
    assert exercise_4_complex_solution(commands, obstacles) == oracle(commands, obstacles)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_stats_solution(commands: List[int], obstacles: List[List[int]]):
    dirs = (0, 1, 0, -1, 0)
    s = {(x, y) for x, y in obstacles}
    x = y = k = 0
    visited = {(0, 0)}
    blocked = 0
    for c in commands:
        if c == -2:
            k = (k + 3) % 4
        elif c == -1:
            k = (k + 1) % 4
        else:
            for _ in range(c):
                nx, ny = x + dirs[k], y + dirs[k + 1]
                if (nx, ny) in s:
                    blocked += 1
                    break
                x, y = nx, ny
                visited.add((x, y))
    return len(visited), blocked

def stats_oracle(commands, obstacles):
    """Mô phỏng độc lập: chữ N/E/S/W và khóa chuỗi."""
    left = {"N": "W", "W": "S", "S": "E", "E": "N"}
    right = {"N": "E", "E": "S", "S": "W", "W": "N"}
    delta = {"N": (0, 1), "E": (1, 0), "S": (0, -1), "W": (-1, 0)}
    blocked_cells = {f"{a},{b}" for a, b in obstacles}
    h, px, py = "N", 0, 0
    seen = ["0,0"]
    blocked_cmds = 0
    for c in commands:
        if c == -2:
            h = left[h]
        elif c == -1:
            h = right[h]
        else:
            hit = False
            for _ in range(c):
                nx, ny = px + delta[h][0], py + delta[h][1]
                if f"{nx},{ny}" in blocked_cells:
                    hit = True
                    break
                px, py = nx, ny
                seen.append(f"{px},{py}")
            blocked_cmds += hit
    return len(set(seen)), blocked_cmds

assert exercise_5_stats_solution([4, -1, 3], []) == (8, 0)
assert exercise_5_stats_solution([4, -1, 4, -2, 4], [[2, 4]]) == (10, 1)
assert exercise_5_stats_solution([6, -1, -1, 6], [[0, 0]]) == (7, 1)
rng = random.Random(5)
for _ in range(400):
    commands = [rng.choice([-2, -1] + list(range(1, 10))) for _ in range(rng.randint(1, 25))]
    obstacles = [[rng.randint(-5, 5), rng.randint(-5, 5)] for _ in range(rng.randint(0, 12))]
    assert exercise_5_stats_solution(commands, obstacles) == stats_oracle(commands, obstacles)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (mô phỏng với `set` và mảng hướng)

```python
class Solution:
    def robotSim(self, commands: List[int], obstacles: List[List[int]]) -> int:
        dirs = (0, 1, 0, -1, 0)                        # bắc, đông, nam, tây: (dirs[k], dirs[k + 1])
        s = {(x, y) for x, y in obstacles}             # tra chướng ngại O(1)
        ans = k = 0
        x = y = 0
        for c in commands:
            if c == -2:                                # quay trái
                k = (k + 3) % 4
            elif c == -1:                              # quay phải
                k = (k + 1) % 4
            else:
                for _ in range(c):                     # đi từng ô một
                    nx, ny = x + dirs[k], y + dirs[k + 1]
                    if (nx, ny) in s:                  # kiểm tra ô KẾ trước khi đi
                        break
                    x, y = nx, ny
                    ans = max(ans, x * x + y * y)      # cập nhật sau MỖI bước
        return ans
```

Công thức nhớ nhanh: `-1 -> (k + 1) % 4`, `-2 -> (k + 3) % 4`; kiểm tra ô kế rồi mới đi; `ans` cập nhật mỗi bước.

### Checklist trước khi submit

- [ ] Chướng ngại lưu trong `set` các tuple (không quét danh sách)?
- [ ] Kiểm tra **ô kế** trước khi di chuyển (không đứng lên chướng ngại)?
- [ ] `-1` là quay **phải**, `-2` là quay **trái**; thứ tự hướng bắc, đông, nam, tây?
- [ ] Cập nhật khoảng cách lớn nhất sau **mỗi bước** (không chỉ ở cuối)?
- [ ] Chướng ngại ở `(0, 0)` được xử lý đúng (bỏ qua lúc xuất phát, chặn khi quay lại)?
- [ ] Đã thử: chướng ngại ngay trước mặt, đi ra rồi quay về, chạy thẳng 45000 bước, toạ độ biên `±3 * 10^4`?
- [ ] Tên method đúng `robotSim(self, commands, obstacles)` và trả `int`?